# ContractIA v20 — Módulo Python

Notebook orquestador que delega todo el trabajo al paquete  (P2 5.1/7.1).

In [ ]:
# Celda 0 — Credenciales + instalación de dependencias
import os, sys, subprocess

NOMBRE_DEL_ARCHIVO_JSON = "agenteia-471917-d588639beeef.json"

def instalar_dependencias():
    paquetes = [
        "langchain", "langchain-core", "langchain-community",
        "langchain-google-vertexai", "pypdf", "docx2txt", "tqdm",
        "pydantic", "networkx", "matplotlib", "tenacity", "nest_asyncio",
    ]
    try:
        import langchain_google_vertexai, docx2txt, pydantic, networkx, tenacity, nest_asyncio
        print("Dependencias ya instaladas.")
    except ImportError:
        print("Instalando dependencias...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + paquetes)
        print("Dependencias instaladas correctamente.")

instalar_dependencias()

if not os.path.exists(NOMBRE_DEL_ARCHIVO_JSON):
    print(f"ERROR: No se encuentra '{NOMBRE_DEL_ARCHIVO_JSON}'.")
else:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = NOMBRE_DEL_ARCHIVO_JSON
    print("Credenciales cargadas.")

In [ ]:
# Celda 1 — Importar módulo contractia
import sys, nest_asyncio
sys.path.insert(0, "src")
nest_asyncio.apply()

import vertexai
from contractia.config import CFG
from contractia.logging_setup import setup_logger
from contractia.carga import procesar_documentos_carpeta
from contractia.metricas import TokenCounterCallback, calcular_metricas_grafo, render_metricas_grafo_md
from contractia.auditoria import ejecutar_auditoria_contrato
from contractia.grafo import construir_indice_nodos_por_cid
from contractia.informe import render_auditoria_markdown
from contractia.chat import iniciar_chat_interactivo
from contractia.main import _build_llm, _save_report

setup_logger()
vertexai.init(project=CFG.project_id, location=CFG.location)
print(f"Vertex AI inicializado. Proyecto: {CFG.project_id}")

In [ ]:
# Celda 2 — Construir LLM y cargar contrato
import time
from IPython.display import display, Markdown

llm = _build_llm()
token_counter = TokenCounterCallback()

docs, texto = procesar_documentos_carpeta(CFG.ruta_contrato)
if not texto:
    raise RuntimeError(f"No se encontraron documentos en '{CFG.ruta_contrato}'.")
print(f"Contrato cargado: {len(texto):,} caracteres.")

In [ ]:
# Celda 3 — Ejecutar pipeline de auditoría
start = time.time()
resultado = ejecutar_auditoria_contrato(texto, llm, callbacks=[token_counter])
elapsed = time.time() - start

print(f"
Auditoría completada en {elapsed:.2f}s")
print("
--- USO DE TOKENS ---")
print(token_counter.resumen())

In [ ]:
# Celda 4 — Generar y guardar informe
md = render_auditoria_markdown(resultado)
md += f"

---
*Tiempo de ejecución: {elapsed:.2f}s*

"
md += f"## Uso de Tokens

{token_counter.resumen()}
"

display(Markdown(md))
_save_report(md)

In [ ]:
# Celda 5 — Métricas de calidad del grafo (P2 2.6.1)
if "grafo" in resultado and not resultado.get("abortado_por_seguridad"):
    mapa = resultado.get("mapa_clausula_a_seccion", {})
    metricas = calcular_metricas_grafo(resultado["grafo"], mapa)
    display(Markdown(render_metricas_grafo_md(metricas)))

In [ ]:
# Celda 6 — Chat interactivo
if "grafo" in resultado and not resultado.get("abortado_por_seguridad"):
    indice_nodos = construir_indice_nodos_por_cid(resultado["grafo"])
    iniciar_chat_interactivo(
        resultado["grafo"],
        resultado["secciones"],
        resultado["indice_secciones"],
        indice_nodos,
        llm,
        callbacks=[token_counter],
    )